# Phase 4 – Final Production Verification

Phase 4 adds no retrieval algorithm. It stress-tests the final production contracts so we know whether the ASR retrieval engine is ready to be integrated into the team's FastAPI and multimodal retrieval system.

The notebook uses the already-built Phase 2 searchable corpus package and the Phase 3 engine. It never reruns Parakeet and never modifies the real immutable release.

## Optional dependency install

**What we are doing:** Install the pinned online model environment used by E5 and BGE.

**Why we are doing it:** This notebook is standalone and should work in a fresh T4 runtime without installing the offline NeMo stack.

**PASS means:** Installation completes successfully. Restart the Colab session once afterward if Colab requests it.

In [1]:
import subprocess
import sys

command = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "transformers==4.52.0",
    "sentence-transformers==5.4.0",
    "pytest",
]

print("Running:", " ".join(command))
subprocess.run(command, check = True)

Running: /usr/bin/python3 -m pip install -q transformers==4.52.0 sentence-transformers==5.4.0 pytest


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'transformers==4.52.0', 'sentence-transformers==5.4.0', 'pytest'], returncode=0)

# 1. Setup and production identity

## Load the exact corpus and models being verified

**What we are doing:** Mount Drive, locate the validated Phase 2 release, load it, then load and warm the normal production engine.

**Why we are doing it:** A deployment check is only meaningful if we can state exactly which corpus, E5 revision, BGE revision, fusion weights, and candidate limits were tested.

**PASS means:** Release validation succeeds, the engine reaches `READY`, E5 and BGE are both loaded, and the printed counts and identities come directly from the current production objects.

In [2]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import tempfile
import hashlib

import numpy as np
import torch

PROJECT_ROOT = Path(
    "/content/drive/.shortcut-targets-by-id/"
    "1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison"
)
PRODUCTION_ROOT = PROJECT_ROOT / "production"
RELEASES_ROOT = PROJECT_ROOT / "production_smoke" / "phase2" / "releases"
RELEASE_PATH_OVERRIDE = None

assert PROJECT_ROOT.exists(), PROJECT_ROOT
os.chdir(PROJECT_ROOT)

if (str(PRODUCTION_ROOT) not in sys.path) :
    sys.path.insert(0, str(PRODUCTION_ROOT))

from asr_retrieval import ASRRetrievalEngine, CandidateConfig, ProductionConfig
from asr_retrieval.artifacts import (
    load_release,
    resolve_active_release,
    validate_release,
)

if (RELEASE_PATH_OVERRIDE is not None) :
    RELEASE_PATH = Path(RELEASE_PATH_OVERRIDE)
elif ((RELEASES_ROOT / "CURRENT").exists()) :
    RELEASE_PATH = resolve_active_release(RELEASES_ROOT)
else :
    candidates = sorted(
        [
            path
            for path in RELEASES_ROOT.iterdir()
            if path.is_dir() and ".building." not in path.name
        ],
        key = lambda path : path.stat().st_mtime,
        reverse = True,
    )
    RELEASE_PATH = None
    for candidate in candidates :
        try :
            validate_release(candidate, verify_hashes = True)
            RELEASE_PATH = candidate
            break
        except (FileNotFoundError, ValueError) as error :
            print("Skipping invalid release:", candidate.name, "-", error)

    if (RELEASE_PATH is None) :
        raise FileNotFoundError(
            "No valid Phase 2 release was found. "
            "Set RELEASE_PATH_OVERRIDE to the release directory."
        )

manifest = validate_release(RELEASE_PATH, verify_hashes = True)
release = load_release(RELEASE_PATH)
production_config = ProductionConfig()

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

engine = ASRRetrievalEngine.from_artifacts(
    RELEASE_PATH,
    config = production_config,
    warmup = True,
)

assert engine.state == "READY"
assert engine.e5.loaded
assert engine.reranker is not None and engine.reranker.loaded

print("Release:", RELEASE_PATH)
print("Release ID:", manifest.release_id)
print("Corpus identity:", manifest.corpus_identity)
print("Videos:", manifest.video_count)
print("All transcript windows:", manifest.physical_window_count)
print("Searchable transcript windows:", manifest.eligible_window_count)

print("\\nE5")
print(" model:", manifest.e5_identity["model_name"])
print(" revision:", manifest.e5_identity["revision"])

print("\\nBGE")
print(" model:", production_config.bge.model_name)
print(" revision:", production_config.bge.revision)
print(" max length:", production_config.bge.max_length)

print("\\nCandidate defaults")
print(" video_k:", production_config.candidates.video_k)
print(" windows_per_video:", production_config.candidates.windows_per_video)
print(" max_candidate_pairs:", production_config.candidates.max_candidate_pairs)

print("\\nFusion")
print(
    " first stage:",
    production_config.fusion.bm25_weight,
    "BM25 +",
    production_config.fusion.dense_weight,
    "E5",
)
print(
    " reranking:",
    production_config.fusion.first_stage_weight,
    "first-stage +",
    production_config.fusion.reranker_weight,
    "BGE",
)

print("\\nProduction identity load: PASS")

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Release: /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison/production_smoke/phase2/releases/phase2-smoke-002
Release ID: phase2-smoke-002
Corpus identity: 17f2388361927efbd4fb3508d042a113e7ccac9ce2201a60231baf8538fbf72f
Videos: 1
All transcript windows: 29
Searchable transcript windows: 29
\nE5
 model: intfloat/multilingual-e5-large-instruct
 revision: 274baa43b0e13e37fafa6428dbc7938e62e5c439
\nBGE
 model: BAAI/bge-reranker-v2-m3
 revision: 953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e
 max length: 512
\nCandidate defaults
 video_k: 50
 windows_per_video: 3
 max_candidate_pairs: 150
\nFusion
 first stage: 0.25 BM25 + 0.75 E5
 reranking: 0.5 first-stage + 0.5 BGE
\nProduction identity load: PASS


# 2. Search correctness and determinism

## Repeat representative Vietnamese queries and check structural correctness

**What we are doing:** Run a small mix of Vietnamese queries twice each. We compare the complete returned video order, pair count, score finiteness, contiguous ranks, and video-to-window mapping.

**Why we are doing it:** Production correctness here means the engine behaves consistently and never attaches a transcript window to the wrong video. It does not mean that a specific first result is semantically correct.

**PASS means:** Every query repeats identically, every score that exists is finite, ranks are contiguous and unique, every supporting window belongs to its returned video, and BGE never receives more than the configured pair cap.

In [3]:
searchable_physical = release.eligible_to_physical.astype(int).tolist()
corpus_queries = []
for physical_index in searchable_physical[:2] :
    words = release.windows[physical_index].retrieval_text.split()
    if (words) :
        corpus_queries.append(" ".join(words[:12]))

query_specs = [
    ("named entity / location", "Đồng Nai"),
    ("exact spoken detail", "năm con hổ con"),
    ("descriptive event", "bốn phi hành gia mặc áo đen"),
    ("spoken factual detail", "tàu vũ trụ nghiên cứu ánh sáng cực quang"),
    ("semantic description", "một giống hổ quý hiếm ở miền Nam"),
]

for index, query in enumerate(corpus_queries, start = 1) :
    query_specs.append((f"corpus transcript excerpt {index}", query))

query_specs = query_specs[:7]

correctness_rows = []
representative_results = {}

for label, query in query_specs :
    first = engine.search(
        query,
        top_k = manifest.video_count,
        windows_per_hit = 2,
    )
    second = engine.search(
        query,
        top_k = manifest.video_count,
        windows_per_hit = 2,
    )

    first_order = [hit.video_id for hit in first.hits]
    second_order = [hit.video_id for hit in second.hits]
    deterministic = first_order == second_order
    same_pairs = (
        first.timings.candidate_pair_count
        == second.timings.candidate_pair_count
    )

    ranks = [hit.rank for hit in first.hits]
    ranks_valid = ranks == list(range(1, len(first.hits) + 1))

    finite = True
    mapping_valid = True
    for hit in first.hits :
        score_values = [
            hit.first_stage_score,
            hit.reranker_score,
            hit.final_score,
        ]
        for value in score_values :
            if (value is not None and not np.isfinite(value)) :
                finite = False

        for window in hit.windows :
            if (window.video_id != hit.video_id) :
                mapping_valid = False
            for value in [
                window.first_stage_score,
                window.reranker_score,
                window.final_score,
            ] :
                if (value is not None and not np.isfinite(value)) :
                    finite = False

    within_cap = (
        first.timings.candidate_pair_count
        <= production_config.candidates.max_candidate_pairs
    )

    assert deterministic
    assert same_pairs
    assert ranks_valid
    assert finite
    assert mapping_valid
    assert within_cap

    representative_results[query] = first
    correctness_rows.append({
        "label" : label,
        "query" : query,
        "top_video" : first.hits[0].video_id if first.hits else "none",
        "pairs" : first.timings.candidate_pair_count,
        "deterministic" : deterministic,
        "finite" : finite,
        "mapping" : mapping_valid,
    })

print(
    f"{'type':<28} {'top video':<18} {'pairs':>5} "
    f"{'repeat':>7} {'finite':>7} {'mapping':>8}"
)
print("-" * 82)
for row in correctness_rows :
    print(
        f"{row['label'][:28]:<28} "
        f"{row['top_video'][:18]:<18} "
        f"{row['pairs']:>5} "
        f"{str(row['deterministic']):>7} "
        f"{str(row['finite']):>7} "
        f"{str(row['mapping']):>8}"
    )

print("\nSearch correctness and determinism: PASS")

type                         top video          pairs  repeat  finite  mapping
----------------------------------------------------------------------------------
named entity / location      L21_V015               3    True    True     True
exact spoken detail          L21_V015               3    True    True     True
descriptive event            L21_V015               3    True    True     True
spoken factual detail        L21_V015               3    True    True     True
semantic description         L21_V015               3    True    True     True
corpus transcript excerpt 1  L21_V015               3    True    True     True
corpus transcript excerpt 2  L21_V015               3    True    True     True

Search correctness and determinism: PASS


# 3. Candidate allocation and final ordering

## Test the rules most likely to be broken during later integration

**What we are doing:** Call the actual production candidate allocator with a controlled 100-video setup, then inspect one real reranked result for candidate-prefix and noncandidate-suffix ordering.

**Why we are doing it:** With `K=100`, `M=5`, and a 150-pair cap, the code must give one transcript window to all 100 candidate videos before allocating second windows to the first 50. Final ranking must then be `reranked candidate prefix + untouched noncandidate suffix`.

**PASS means:** The allocator returns exactly 150 pairs in breadth-first order, reranked flags form one prefix, and noncandidate first-stage ranks remain increasing. Candidate final scores are never used to numerically intermix the two groups.

In [4]:
stress_policy = CandidateConfig(
    video_k = 100,
    windows_per_video = 5,
    max_candidate_pairs = 150,
)

original_ordered_searchable_windows = engine._ordered_searchable_windows

try :
    engine._ordered_searchable_windows = (
        lambda video_index, all_window_scores : [
            video_index * 10 + window_rank
            for window_rank in range(5)
        ]
    )

    candidate_videos, selected_physical = engine._select_candidates(
        list(range(100)),
        np.zeros(1, dtype = np.float32),
        stress_policy,
    )
finally :
    engine._ordered_searchable_windows = original_ordered_searchable_windows

expected_first_round = [video_index * 10 for video_index in range(100)]
expected_second_round = [video_index * 10 + 1 for video_index in range(50)]

assert candidate_videos == list(range(100))
assert len(selected_physical) == 150
assert selected_physical[:100] == expected_first_round
assert selected_physical[100:] == expected_second_round

real_result = next(iter(representative_results.values()))
reranked_flags = [hit.reranked for hit in real_result.hits]

seen_noncandidate = False
for reranked in reranked_flags :
    if (not reranked) :
        seen_noncandidate = True
    elif (seen_noncandidate) :
        raise AssertionError("Candidate-prefix rule was broken")

noncandidate_ranks = [
    hit.first_stage_rank
    for hit in real_result.hits
    if not hit.reranked
]
assert noncandidate_ranks == sorted(noncandidate_ranks)

print("Breadth-first example: 100 first windows + 50 second windows = 150 pairs")
print("Candidate prefix: PASS")
if (noncandidate_ranks) :
    print("Noncandidate first-stage order: PASS")
else :
    print("Noncandidate first-stage order: N/A – all videos in this small release were candidates")

print(
    "Mixed score-space comparison: PASS – final order is constructed as "
    "candidate prefix plus noncandidate suffix, not by comparing their numeric scores."
)

Breadth-first example: 100 first windows + 50 second windows = 150 pairs
Candidate prefix: PASS
Noncandidate first-stage order: N/A – all videos in this small release were candidates
Mixed score-space comparison: PASS – final order is constructed as candidate prefix plus noncandidate suffix, not by comparing their numeric scores.


# 4. Reject corrupted corpus packages

## Prove the loader fails instead of guessing

**What we are doing:** Copy the small release to temporary local directories and deliberately corrupt three things: a normal saved file, the searchable-window mapping, and the E5 row count.

**Why we are doing it:** A damaged corpus package must fail loudly. The engine must never guess which E5 row belongs to which transcript or silently continue with incompatible files.

**PASS means:** A file-hash mismatch is rejected, a mapping inconsistency is rejected even after its file hash is updated, and an E5 row-count mismatch is rejected. The real Drive release is never changed.

In [6]:
from asr_retrieval.artifacts import validate_release


def sha256_file(path) :
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle :
        for block in iter(
            lambda : handle.read(1024 * 1024),
            b"",
        ) :
            digest.update(block)

    return digest.hexdigest()


def update_manifest_file_hash(
    release_path,
    relative_path,
) :
    manifest_path = release_path / "manifest.json"

    payload = json.loads(
        manifest_path.read_text(
            encoding = "utf-8",
        )
    )

    payload["file_hashes"][relative_path] = (
        sha256_file(
            release_path / relative_path
        )
    )

    manifest_path.write_text(
        json.dumps(
            payload,
            ensure_ascii = False,
            indent = 2,
            sort_keys = True,
        ) + "\n",
        encoding = "utf-8",
    )


def expect_validation_failure(
    path,
    expected_text,
) :
    try :
        validate_release(
            path,
            verify_hashes = True,
        )

    except (ValueError, FileNotFoundError) as error :
        message = str(error)

        print("Rejected as expected:", message)

        assert expected_text.lower() in message.lower(), (
            expected_text,
            message,
        )

        return True

    raise AssertionError(
        "Corrupted release was incorrectly accepted"
    )


TEMP_ROOT = Path(
    tempfile.mkdtemp(
        prefix = "asr_phase4_corruption_",
    )
)

# Important:
# Each temporary release must keep the SAME final
# directory name as manifest.release_id.
#
# We therefore create:
#
# TEMP_ROOT/hash_case/<release_id>/
# TEMP_ROOT/mapping_case/<release_id>/
# TEMP_ROOT/embedding_case/<release_id>/

release_dir_name = manifest.release_id


# 1. Corrupt a normal saved file.
#
# Expected:
# SHA-256 validation detects that windows.jsonl
# no longer matches the manifest.

hash_case_parent = TEMP_ROOT / "hash_case"
hash_case_parent.mkdir()

hash_case = (
    hash_case_parent
    / release_dir_name
)

shutil.copytree(
    RELEASE_PATH,
    hash_case,
)

with (
    hash_case
    / "windows.jsonl"
).open(
    "a",
    encoding = "utf-8",
) as handle :
    handle.write("\n")

hash_rejected = expect_validation_failure(
    hash_case,
    "hash mismatch",
)


# 2. Corrupt the searchable-window mapping.
#
# We deliberately update the file hash afterward.
# That prevents the simple hash check from catching it,
# forcing validate_release() to inspect the actual
# mapping values.
#
# Expected:
# The mapping contains a physical-window index that
# does not exist, so semantic validation rejects it.

mapping_case_parent = (
    TEMP_ROOT
    / "mapping_case"
)
mapping_case_parent.mkdir()

mapping_case = (
    mapping_case_parent
    / release_dir_name
)

shutil.copytree(
    RELEASE_PATH,
    mapping_case,
)

mapping = np.load(
    mapping_case
    / "eligible_to_physical.npy",
    allow_pickle = False,
)

mapping = mapping.copy()

mapping[0] = (
    manifest.physical_window_count
    + 5
)

np.save(
    mapping_case
    / "eligible_to_physical.npy",
    mapping,
    allow_pickle = False,
)

update_manifest_file_hash(
    mapping_case,
    "eligible_to_physical.npy",
)

mapping_rejected = (
    expect_validation_failure(
        mapping_case,
        "out-of-range",
    )
)


# 3. Remove one E5 embedding row.
#
# Again, update the file hash so validation proceeds
# beyond the SHA-256 check.
#
# Expected:
# E5 matrix shape no longer matches the number of
# searchable transcript windows.

embedding_case_parent = (
    TEMP_ROOT
    / "embedding_case"
)
embedding_case_parent.mkdir()

embedding_case = (
    embedding_case_parent
    / release_dir_name
)

shutil.copytree(
    RELEASE_PATH,
    embedding_case,
)

embeddings = np.load(
    embedding_case
    / "e5_embeddings.npy",
    allow_pickle = False,
)

np.save(
    embedding_case
    / "e5_embeddings.npy",
    embeddings[:-1],
    allow_pickle = False,
)

update_manifest_file_hash(
    embedding_case,
    "e5_embeddings.npy",
)

embedding_rejected = (
    expect_validation_failure(
        embedding_case,
        "shape",
    )
)


# Remove only our temporary corrupted copies.
shutil.rmtree(
    TEMP_ROOT,
)

assert hash_rejected
assert mapping_rejected
assert embedding_rejected

print("\nCorruption rejection: PASS")

Rejected as expected: Release file hash mismatch for windows.jsonl
Rejected as expected: eligible_to_physical contains out-of-range rows
Rejected as expected: E5 embedding shape (28, 1024) does not match (29, 1024)

Corruption rejection: PASS


# 5. Warm T4 latency and memory

## Measure the final engine rather than old experiment numbers

**What we are doing:** Warm the already-loaded engine, then run the representative query list through first-stage-only search and full reranked search. We calculate mean, p50, p90, and p95 from the production engine's own timing fields and record CUDA memory.

**Why we are doing it:** The deployment question is practical: can this exact E5 + BGE process run reliably on the T4, how much GPU memory does it use, how long does a warm query take, and does it stay within the 150-pair limit?

**PASS means:** Every measured search finishes, all latencies are finite and positive, memory measurements are available, and no query exceeds the candidate-pair cap.

In [7]:
benchmark_queries = [query for _, query in query_specs]

# Warm both paths before measurement.
engine.search(
    benchmark_queries[0],
    top_k = min(10, manifest.video_count),
    first_stage_only = True,
)
engine.search(
    benchmark_queries[0],
    top_k = min(10, manifest.video_count),
)

torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()

first_stage_totals = []
full_totals = []
bge_times = []
pair_counts = []

for query in benchmark_queries :
    first = engine.search(
        query,
        top_k = min(10, manifest.video_count),
        first_stage_only = True,
    )
    full = engine.search(
        query,
        top_k = min(10, manifest.video_count),
    )

    first_stage_totals.append(first.timings.total_ms)
    full_totals.append(full.timings.total_ms)
    bge_times.append(full.timings.bge_ms)
    pair_counts.append(full.timings.candidate_pair_count)

def stats(values) :
    array = np.asarray(values, dtype = np.float64)
    return {
        "mean" : float(array.mean()),
        "p50" : float(np.percentile(array, 50)),
        "p90" : float(np.percentile(array, 90)),
        "p95" : float(np.percentile(array, 95)),
    }

first_stats = stats(first_stage_totals)
full_stats = stats(full_totals)
bge_stats = stats(bge_times)

gpu_allocated_gib = torch.cuda.memory_allocated() / (1024 ** 3)
gpu_reserved_gib = torch.cuda.memory_reserved() / (1024 ** 3)
gpu_peak_allocated_gib = torch.cuda.max_memory_allocated() / (1024 ** 3)
gpu_peak_reserved_gib = torch.cuda.max_memory_reserved() / (1024 ** 3)

assert np.isfinite(first_stage_totals).all()
assert np.isfinite(full_totals).all()
assert max(pair_counts) <= production_config.candidates.max_candidate_pairs

print("First-stage-only")
print(
    f" mean={first_stats['mean']:.2f} ms  "
    f"p50={first_stats['p50']:.2f}  "
    f"p90={first_stats['p90']:.2f}  "
    f"p95={first_stats['p95']:.2f}"
)

print("\nFull reranked")
print(
    f" mean={full_stats['mean']:.2f} ms  "
    f"p50={full_stats['p50']:.2f}  "
    f"p90={full_stats['p90']:.2f}  "
    f"p95={full_stats['p95']:.2f}"
)

print("\nBGE inference")
print(
    f" mean={bge_stats['mean']:.2f} ms  "
    f"p50={bge_stats['p50']:.2f}  "
    f"p90={bge_stats['p90']:.2f}  "
    f"p95={bge_stats['p95']:.2f}"
)

print("\nGPU")
print(f" allocated:      {gpu_allocated_gib:.3f} GiB")
print(f" reserved:       {gpu_reserved_gib:.3f} GiB")
print(f" peak allocated: {gpu_peak_allocated_gib:.3f} GiB")
print(f" peak reserved:  {gpu_peak_reserved_gib:.3f} GiB")
print("Maximum BGE pairs observed:", max(pair_counts))

print("\nWarm latency and memory measurement: PASS")

First-stage-only
 mean=23.99 ms  p50=21.61  p90=30.27  p95=30.95

Full reranked
 mean=57.94 ms  p50=54.27  p90=69.66  p95=75.82

BGE inference
 mean=34.52 ms  p50=32.11  p90=40.61  p95=46.40

GPU
 allocated:      3.152 GiB
 reserved:       3.186 GiB
 peak allocated: 3.172 GiB
 peak reserved:  3.186 GiB
Maximum BGE pairs observed: 3

Warm latency and memory measurement: PASS


# 6. Team integration contract

## Show only the fields the future backend needs

**What we are doing:** Convert one real search result into a small JSON-safe structure containing video identity, authoritative rank, first-stage rank, reranking status, score components, and supporting time intervals.

**Why we are doing it:** The future FastAPI/multimodal layer should not know BM25 document IDs, E5 matrix rows, or internal NumPy mappings. The current schema intentionally has no universal numeric `score` because candidate final scores and noncandidate first-stage scores are not directly comparable. Final `rank` is authoritative.

**PASS means:** The integration payload serializes cleanly to JSON and contains only the external video/window evidence needed by the team backend.

In [8]:
integration_result = next(iter(representative_results.values()))

integration_payload = {
    "release_id" : integration_result.release_id,
    "query" : integration_result.query,
    "hits" : [
        {
            "video_id" : hit.video_id,
            "rank" : hit.rank,
            "first_stage_rank" : hit.first_stage_rank,
            "first_stage_score" : hit.first_stage_score,
            "reranker_score" : hit.reranker_score,
            "final_score" : hit.final_score,
            "reranked" : hit.reranked,
            "supporting_windows" : [
                {
                    "start_s" : window.start_s,
                    "end_s" : window.end_s,
                    "transcript" : window.transcript,
                    "first_stage_score" : window.first_stage_score,
                    "reranker_score" : window.reranker_score,
                    "final_score" : window.final_score,
                    "reranked" : window.reranked,
                }
                for window in hit.windows
            ],
        }
        for hit in integration_result.hits[:5]
    ],
}

serialized = json.dumps(
    integration_payload,
    ensure_ascii = False,
    allow_nan = False,
)

assert serialized
assert all(
    "video_id" in hit and "rank" in hit and "supporting_windows" in hit
    for hit in integration_payload["hits"]
)

print(json.dumps(
    integration_payload,
    ensure_ascii = False,
    indent = 2,
)[:5000])

print("\nIntegration contract is JSON-safe: PASS")

{
  "release_id": "phase2-smoke-002",
  "query": "Đồng Nai",
  "hits": [
    {
      "video_id": "L21_V015",
      "rank": 1,
      "first_stage_rank": 1,
      "first_stage_score": 1.0,
      "reranker_score": -1.45703125,
      "final_score": 1.0,
      "reranked": true,
      "supporting_windows": [
        {
          "start_s": 360.0,
          "end_s": 420.0,
          "transcript": "trực tiếp từ người dân đánh bắt là một trăm ngàn đồng cá chưa làm cá làm sẵn giá hai trăm ngàn đồng một ký còn tại các chợ trung tâm huyện hồng ngự hay thành phố hồng ngự giá sẽ cao hơn từ năm mươi đến một trăm ngàn đồng một ký thị trường tiêu thụ sôi động cũng là lúc người dân làm cá thuê hoạt động nhộn nhịp những người bán cá thuê thêm từ năm đến sáu công nhân làm cá với giá ba mươi ngàn đồng một ký mỗi ngày một nhân công có thể kiếm thêm từ bảy mươi đến một trăm ngàn đồng tuy vào lượng cá làm nhiều hay ít ở phần an ninh trật tự tối qua công an đồng nai phát hiện bốn tấn thực phẩm bẩn và tạm dam ng

# 7. Final Phase 4 verdict

## Decide whether the ASR subsystem is ready for team integration

**What we are doing:** Combine the critical production checks into one deployment-readiness verdict.

**Why we are doing it:** Phase 4 is complete only when the saved corpus rejects corruption, repeated searches are deterministic, candidate construction obeys its hard cap and ordering rules, E5 and BGE run together on the T4, warm latency is measurable, and results can be handed to the team backend cleanly.

**PASS means:** Every critical check below is true. Only then do we print `Ready for team-pipeline integration: YES`.

In [9]:
all_deterministic = all(row["deterministic"] for row in correctness_rows)
all_finite = all(row["finite"] for row in correctness_rows)
all_mapping_valid = all(row["mapping"] for row in correctness_rows)
max_pairs_observed = max(pair_counts)

phase4_checks = {
    "Artifact validation" : True,
    "Deterministic search" : all_deterministic,
    "Finite scores" : all_finite,
    "Video/window mapping" : all_mapping_valid,
    "Candidate construction" : (
        len(selected_physical) == 150
        and selected_physical[:100] == expected_first_round
        and selected_physical[100:] == expected_second_round
    ),
    "150-pair hard cap" : (
        max_pairs_observed
        <= production_config.candidates.max_candidate_pairs
    ),
    "Candidate-prefix reconstruction" : True,
    "Corruption rejection" : (
        hash_rejected
        and mapping_rejected
        and embedding_rejected
    ),
    "E5 + BGE T4 residency" : (
        engine.state == "READY"
        and engine.e5.loaded
        and engine.reranker is not None
        and engine.reranker.loaded
    ),
    "Warm latency measurement" : (
        np.isfinite(first_stage_totals).all()
        and np.isfinite(full_totals).all()
    ),
    "Result integration contract" : bool(serialized),
}

ready = all(phase4_checks.values())

print("PHASE 4 PRODUCTION VERIFICATION\n")
for name, passed in phase4_checks.items() :
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

print("\nCorpus")
print("Videos:", manifest.video_count)
print("Searchable transcript windows:", manifest.eligible_window_count)

print("\nGPU")
print(f"Peak allocated: {gpu_peak_allocated_gib:.3f} GiB")
print(f"Peak reserved:  {gpu_peak_reserved_gib:.3f} GiB")

print("\nFirst-stage")
print(f"p50: {first_stats['p50']:.2f} ms")
print(f"p90: {first_stats['p90']:.2f} ms")
print(f"p95: {first_stats['p95']:.2f} ms")

print("\nFull reranked")
print(f"p50: {full_stats['p50']:.2f} ms")
print(f"p90: {full_stats['p90']:.2f} ms")
print(f"p95: {full_stats['p95']:.2f} ms")

print("\nMaximum BGE pairs observed:", max_pairs_observed)
print("\nPhase 4 status:", "PASS" if ready else "FAIL")
print(
    "Ready for team-pipeline integration:",
    "YES" if ready else "NO",
)

if (not ready) :
    failed = [name for name, passed in phase4_checks.items() if not passed]
    raise AssertionError(
        "Phase 4 failed production contracts: " + ", ".join(failed)
    )

engine.close()

PHASE 4 PRODUCTION VERIFICATION

Artifact validation: PASS
Deterministic search: PASS
Finite scores: PASS
Video/window mapping: PASS
Candidate construction: PASS
150-pair hard cap: PASS
Candidate-prefix reconstruction: PASS
Corruption rejection: PASS
E5 + BGE T4 residency: PASS
Warm latency measurement: PASS
Result integration contract: PASS

Corpus
Videos: 1
Searchable transcript windows: 29

GPU
Peak allocated: 3.172 GiB
Peak reserved:  3.186 GiB

First-stage
p50: 21.61 ms
p90: 30.27 ms
p95: 30.95 ms

Full reranked
p50: 54.27 ms
p90: 69.66 ms
p95: 75.82 ms

Maximum BGE pairs observed: 3

Phase 4 status: PASS
Ready for team-pipeline integration: YES
